# Device tampering and physical compromise  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 4.1.b: Device tampering and physical compromise

**Skill:** Analyze Risk · **Maps to:** 4.1.B · **Language:** Python 3 (pytest)

## Problem
In a physical security audit, we collect logs from system motherboards, BIOS events, and OS events. You must implement a parser that reads these logs and identifies all compromised devices.

A device is considered physically compromised if it triggers any of the following rules:
1.  **Chassis Intrusion:** `event_type == "chassis_intrusion"` and `details` contains `"open"` (case-insensitive).
2.  **Unauthorized Boot:** `event_type == "boot"` and `details` contains `"usb"`, `"external"`, or `"live"` (case-insensitive).
3.  **USB Rubber Ducky:** A `usb_insert` event with `"keyboard"` or `"hid"` in `details` (case-insensitive) followed by a `command` event on the same device within 3 seconds, where the command `details` contains `"powershell"`, `"cmd.exe"`, or `"download"` (case-insensitive).

## Function contract
Implement `solve(logs: list) -> list` in the code editor (see the Reference section below).
Return the sorted list of compromised device IDs.

## Input
A list of dicts:
```python
[
    {
        "timestamp": "2026-06-25T12:00:00Z",
        "device_id": "laptop_01",
        "event_type": "boot",
        "details": "Booted from external USB drive"
    }
]
```

## Output
A list of strings, e.g. `["laptop_01"]`, sorted alphabetically.

## Examples
```python
solve([
    {"timestamp": "2026-06-25T10:00:00Z", "device_id": "pc_01", "event_type": "chassis_intrusion", "details": "chassis opened detected"}
]) -> ["pc_01"]

solve([
    {"timestamp": "2026-06-25T10:00:00Z", "device_id": "pc_02", "event_type": "usb_insert", "details": "USB Keyboard connected"},
    {"timestamp": "2026-06-25T10:00:02Z", "device_id": "pc_02", "event_type": "command", "details": "powershell -nop -w hidden -c Get-Process"}
]) -> ["pc_02"]
```

## Constraints
- Pure function, offline, standard library only.
- Timestamps are ISO-8601 strings.
- Sorted return list with unique elements.

## Reference
Any implementation that satisfies this interface must pass the tests unchanged.

### Entry point
```python
def solve(logs: list[dict]) -> list[str]
```

### Input (`logs`)
A list of log dictionaries, sorted chronologically. Each log contains:
- `timestamp`: ISO-8601 string (e.g. `"2026-06-25T12:00:00Z"`)
- `device_id`: string identifier (e.g. `"workstation_01"`)
- `event_type`: string representing the log event type
- `details`: string containing incident or event details

### Output (`result`)
A list of strings representing the `device_id`s flagged as physically compromised, sorted alphabetically with no duplicates.

### Detection Rules (Case-Insensitive)
1.  **Chassis Intrusion:** Any log entry where `event_type == "chassis_intrusion"` and the word `"open"` is present in the `details`.
2.  **Unauthorized Boot:** Any log entry where `event_type == "boot"` and the `details` contains the words `"usb"`, `"external"`, or `"live"`.
3.  **USB Rubber Ducky (Keystroke Injection):** A log entry where `event_type == "usb_insert"` and `details` contains `"keyboard"` or `"hid"`, which is followed by a `command` log entry on the *same* device such that:
    - The command's timestamp `t_cmd` is greater than or equal to the USB insertion's timestamp `t_usb`.
    - The time difference `(t_cmd - t_usb).total_seconds() <= 3`.
    - The command's `details` contains the words `"powershell"`, `"cmd.exe"`, or `"download"`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- Timestamps can be parsed using `datetime.fromisoformat(ts.replace("Z", "+00:00"))`.


In [ ]:
# Reference solution
"""Reference solution for 4.1.b. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from datetime import datetime


def solve(logs: list) -> list:
    """Analyze logs to detect physical tampering and device compromise.

    Args:
        logs: A list of log dictionaries, sorted chronologically.

    Returns:
        A list of compromised device IDs, sorted alphabetically.
    """
    compromised = set()
    if not logs:
        return []

    def parse_time(ts_str):
        return datetime.fromisoformat(ts_str.replace("Z", "+00:00"))

    # For fast lookup of USB insertions per device
    usb_inserts = []  # List of (timestamp_dt, device_id)

    # Process all events
    for event in logs:
        device_id = event.get("device_id")
        event_type = event.get("event_type", "")
        details = event.get("details", "")
        ts_str = event.get("timestamp", "")

        if not device_id or not ts_str:
            continue

        try:
            ts_dt = parse_time(ts_str)
        except ValueError:
            continue

        details_lower = details.lower()

        # Rule 1: Chassis Intrusion
        if event_type == "chassis_intrusion" and "open" in details_lower:
            compromised.add(device_id)

        # Rule 2: Unauthorized Boot
        elif event_type == "boot" and any(k in details_lower for k in ["usb", "external", "live"]):
            compromised.add(device_id)

        # Record USB insert for Rule 3
        elif event_type == "usb_insert" and any(k in details_lower for k in ["keyboard", "hid"]):
            usb_inserts.append((ts_dt, device_id))

        # Check Command for Rule 3
        elif event_type == "command":
            for usb_ts, usb_dev in usb_inserts:
                if usb_dev == device_id:
                    time_diff = (ts_dt - usb_ts).total_seconds()
                    if 0 <= time_diff <= 3:
                        if any(k in details_lower for k in ["powershell", "cmd.exe", "download"]):
                            compromised.add(device_id)

    return sorted(list(compromised))


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def test_empty():
    assert impl.solve([]) == []


def test_chassis_intrusion():
    logs = [
        {"timestamp": "2026-06-25T10:00:00Z", "device_id": "pc_01", "event_type": "chassis_intrusion", "details": "chassis opened detected"}
    ]
    assert impl.solve(logs) == ["pc_01"]


def test_unauthorized_boot():
    logs = [
        {"timestamp": "2026-06-25T10:00:00Z", "device_id": "laptop_02", "event_type": "boot", "details": "Booted from external Live USB"}
    ]
    assert impl.solve(logs) == ["laptop_02"]


def test_usb_rubber_ducky_success():
    logs = [
        {"timestamp": "2026-06-25T10:00:00Z", "device_id": "pc_02", "event_type": "usb_insert", "details": "USB HID Keyboard connected"},
        {"timestamp": "2026-06-25T10:00:02Z", "device_id": "pc_02", "event_type": "command", "details": "powershell -nop -w hidden -c payload"}
    ]
    assert impl.solve(logs) == ["pc_02"]


def test_usb_rubber_ducky_too_late():
    logs = [
        {"timestamp": "2026-06-25T10:00:00Z", "device_id": "pc_02", "event_type": "usb_insert", "details": "USB HID Keyboard connected"},
        {"timestamp": "2026-06-25T10:00:05Z", "device_id": "pc_02", "event_type": "command", "details": "powershell -nop -w hidden -c payload"}
    ]
    assert impl.solve(logs) == []


def test_usb_rubber_ducky_different_device():
    logs = [
        {"timestamp": "2026-06-25T10:00:00Z", "device_id": "pc_01", "event_type": "usb_insert", "details": "USB HID Keyboard connected"},
        {"timestamp": "2026-06-25T10:00:01Z", "device_id": "pc_02", "event_type": "command", "details": "powershell -nop -w hidden -c payload"}
    ]
    assert impl.solve(logs) == []


def test_multiple_compromises():
    logs = [
        {"timestamp": "2026-06-25T10:00:00Z", "device_id": "pc_03", "event_type": "chassis_intrusion", "details": "Intrusion switch opened"},
        {"timestamp": "2026-06-25T10:05:00Z", "device_id": "pc_01", "event_type": "boot", "details": "booting usb medium"},
        {"timestamp": "2026-06-25T10:10:00Z", "device_id": "pc_02", "event_type": "user_login", "details": "admin login"}
    ]
    assert impl.solve(logs) == ["pc_01", "pc_03"]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
